# Embeddings with EmbeddingGemma 2

The same 2,100 UC Merced scenes as assignment 1, embedded with a multimodal model instead
of a vision transformer.

Output: `data/embeddinggemma_uc_merced.npz`

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

## Load the model

`bfloat16` or `float32`, never `float16`.

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

print (dtype)

model = SentenceTransformer("google/embeddinggemma-2", model_kwargs={"torch_dtype": dtype})

print(dtype, "|", model.get_embedding_dimension())

## The dataset

2,100 aerial scenes, 21 classes, 256 x 256.

In [ ]:
ds = load_dataset("blanchon/UC_Merced", split="train")

class_names = ds.features["label"].names

y = np.array(ds["label"])
imgs = [r["image"].convert("RGB") for r in ds]

print(len(imgs), "images |", len(class_names), "classes")

fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))

for ax, i in zip(axes, range(8)):
    ax.imshow(imgs[i])
    ax.set_title(class_names[y[i]], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Embed everything

One vector per image

In [ ]:
EMB = Path("data/embeddinggemma_uc_merced.npz")
EMB.parent.mkdir(exist_ok=True)

if EMB.exists():
    d = np.load(EMB)
    E = d["E"]
    print(f"loaded {EMB} - skipped")
else:
    import time

    t0 = time.perf_counter()
    E = model.encode([{"image": im} for im in imgs], batch_size=8, normalize_embeddings=True)
    dt = time.perf_counter() - t0

    np.savez_compressed(EMB, idx=np.arange(len(imgs)), y=y,
                        class_names=np.array(class_names), E=E.astype(np.float32))
    print(f"saved {EMB}  ({dt:.1f}s for {len(imgs)} images)")

print(E.shape, E.dtype)

## Sanity check

The model has never seen these 21 classes. Compare one image against the class names.

In [ ]:
i = 0

T = model.encode([f"an aerial photo of {c}" for c in class_names],
                 normalize_embeddings=True)

scores = E[i] @ T.T

for rank in np.argsort(-scores)[:3]:
    print(f"{scores[rank]:+.3f}  {class_names[rank]}")

print(f"\ntrue class: {class_names[y[i]]}")

In [ ]:
q = np.random.default_rng().integers(len(E))

sims = E @ E[q]
sims[q] = -np.inf

nearest = np.argsort(-sims)[:19]

fig, axes = plt.subplots(4, 5, figsize=(13, 11))

for ax in axes.ravel():
    ax.axis("off")

axes[0, 0].imshow(imgs[q])
axes[0, 0].set_title(f"query\n{class_names[y[q]]}", fontsize=8)

for ax, j in zip(axes.ravel()[1:], nearest):
    ax.imshow(imgs[j])
    ax.set_title(f"{class_names[y[j]]}\n{sims[j]:+.2f}", fontsize=8)

plt.tight_layout()
plt.show()